# K-means — Ejemplos

**Módulo 3 — Modelado y evaluación de enfoques no supervisados · Curso Analítica de Datos**

Este notebook acompaña las diapositivas [`3.9_Kmeans_agrupamiento_basado_en_centroides.pdf`](3.9_Kmeans_agrupamiento_basado_en_centroides.pdf) y lleva a código lo que allí se explica.

## Contenido

1. **Una iteración de K-means, a mano**: reproducimos exactamente la actividad de las diapositivas (datos A, B, C, D) — funciona como hoja de respuestas.
2. **El ciclo completo, desde cero**: implementamos el algoritmo de Lloyd-Forgy y visualizamos cómo se mueven los centroides iteración a iteración.
3. **Inercia y el método del codo.**
4. **Coeficiente silhouette.**
5. **Inicialización**: por qué `k-means++` es preferible a centroides aleatorios.
6. **Alcance y limitaciones**: qué pasa con grupos no convexos y con valores atípicos.

> 💡 Este notebook **no requiere descargar nada ni tener cuenta de Kaggle**: todos los datasets se generan en el propio notebook.

In [ ]:
# Librerías que usaremos en todo el notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, silhouette_samples

pd.set_option('display.max_columns', 20)
plt.rcParams['figure.figsize'] = (7, 5)
rng = np.random.default_rng(42)

print('Librerías cargadas correctamente ✅')

---
## 1. Una iteración de K-means, a mano

Reproducimos la actividad de la diapositiva 9: cuatro datos de "horas de estudio por semana" — A=1, B=2, C=8, D=9 — con $K=2$ y centroides iniciales $C_1=1$, $C_2=9$.

In [ ]:
datos = pd.DataFrame({'punto': ['A', 'B', 'C', 'D'], 'x': [1, 2, 8, 9]})
C1_inicial, C2_inicial = 1, 9

# Paso 1 (asignar): cada dato al centroide más cercano, con distancia absoluta |x - C|
datos['dist_C1'] = (datos['x'] - C1_inicial).abs()
datos['dist_C2'] = (datos['x'] - C2_inicial).abs()
datos['grupo'] = np.where(datos['dist_C1'] <= datos['dist_C2'], 'Grupo 1', 'Grupo 2')
datos

In [ ]:
# Paso 2 (actualizar): cada nuevo centroide es el promedio de su grupo
C1_nuevo = datos.loc[datos['grupo'] == 'Grupo 1', 'x'].mean()
C2_nuevo = datos.loc[datos['grupo'] == 'Grupo 2', 'x'].mean()
print(f'C1 = promedio({list(datos.loc[datos["grupo"]=="Grupo 1","x"])}) = {C1_nuevo}')
print(f'C2 = promedio({list(datos.loc[datos["grupo"]=="Grupo 2","x"])}) = {C2_nuevo}')

In [ ]:
# Paso 3: inercia J2 con K=2, usando los centroides YA actualizados (como pide la actividad)
J2 = ((datos.loc[datos['grupo'] == 'Grupo 1', 'x'] - C1_nuevo) ** 2).sum() + \
     ((datos.loc[datos['grupo'] == 'Grupo 2', 'x'] - C2_nuevo) ** 2).sum()
print(f'J2 = (1-{C1_nuevo})² + (2-{C1_nuevo})² + (8-{C2_nuevo})² + (9-{C2_nuevo})² = {J2}')

### Completando la tabla de la actividad: inercia para K = 1, 2, 3 y 4

La diapositiva ya da $J_1=50$, $J_3=0{,}5$ y $J_4=0$, y deja $J_2$ en blanco — que es justo lo que acabamos de calcular a mano. Verificamos los cuatro valores con `scikit-learn`.

In [ ]:
X_1d = datos[['x']].values

print(f'{"K":>3s} {"Inercia":>10s}')
for k in [1, 2, 3, 4]:
    modelo_k = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X_1d)
    print(f'{k:3d} {modelo_k.inertia_:10.1f}')

print('\n¿Por qué no basta con buscar una inercia igual a cero? Porque con K=4 (un grupo por cada')
print('dato) la inercia siempre da 0 — eso no agrupa nada, solo memoriza los datos. El método del')
print('codo busca el punto donde seguir aumentando K deja de reducir la inercia de forma importante.')

---
## 2. El ciclo completo de K-means, desde cero

Implementamos las tres reglas de la diapositiva 3 (inicializar, asignar, actualizar) sobre un dataset 2D con 3 grupos, y visualizamos cómo se mueven los centroides en cada iteración — igual que la secuencia a-f de las diapositivas.

In [ ]:
def kmeans_desde_cero(X, k, semilla=0, max_iter=10):
    """Implementación mínima del ciclo de Lloyd-Forgy: inicializar, asignar, actualizar, repetir."""
    generador = np.random.default_rng(semilla)
    # 1) Inicializar: k instancias reales del propio dataset, elegidas al azar
    indices_iniciales = generador.choice(len(X), size=k, replace=False)
    centroides = X[indices_iniciales].copy()
    historial = [centroides.copy()]

    for _ in range(max_iter):
        # 2) Asignar cada punto al centroide más cercano (distancia euclídea al cuadrado)
        distancias = np.linalg.norm(X[:, None, :] - centroides[None, :, :], axis=2)
        asignaciones = distancias.argmin(axis=1)

        # 3) Actualizar cada centroide como el promedio de los puntos que se le asignaron
        nuevos_centroides = np.array([
            X[asignaciones == j].mean(axis=0) if np.any(asignaciones == j) else centroides[j]
            for j in range(k)
        ])
        historial.append(nuevos_centroides.copy())

        if np.allclose(nuevos_centroides, centroides):
            break  # los centroides dejaron de moverse: convergencia
        centroides = nuevos_centroides

    return centroides, asignaciones, historial

X_grupos, _ = make_blobs(n_samples=150, centers=3, cluster_std=1.0, random_state=3)
centroides_finales, asignaciones_finales, historial = kmeans_desde_cero(X_grupos, k=3, semilla=1)

print(f'Convergió en {len(historial) - 1} iteraciones.')
print('Centroides finales:\n', np.round(centroides_finales, 2))

In [ ]:
n_paneles = min(4, len(historial))
fig, axes = plt.subplots(1, n_paneles, figsize=(4 * n_paneles, 4))
for i, ax in enumerate(axes):
    centroides_i = historial[i]
    distancias_i = np.linalg.norm(X_grupos[:, None, :] - centroides_i[None, :, :], axis=2)
    asignaciones_i = distancias_i.argmin(axis=1)
    ax.scatter(X_grupos[:, 0], X_grupos[:, 1], c=asignaciones_i, cmap='tab10', s=12)
    ax.scatter(centroides_i[:, 0], centroides_i[:, 1], c='black', marker='x', s=120)
    ax.set_title(f'Iteración {i}')
plt.tight_layout()
plt.show()

print('Observa cómo los centroides (X negras) arrancan en puntos reales del dataset y se van')
print('desplazando hacia el centro de su grupo en cada iteración, hasta dejar de moverse.')

---
## 3. Inercia y el método del codo

La inercia ($J$, diapositiva 6) **nunca aumenta** al agregar más grupos. El método del codo busca el punto donde seguir aumentando $K$ deja de ayudar mucho.

In [ ]:
rango_k = range(1, 9)
inercias = []
for k in rango_k:
    modelo = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X_grupos)
    inercias.append(modelo.inertia_)

fig, ax = plt.subplots()
ax.plot(list(rango_k), inercias, marker='o')
ax.axvline(3, linestyle='--', color='gray', alpha=0.7)
ax.set_xlabel('K (número de grupos)')
ax.set_ylabel('Inercia')
ax.set_title('Método del codo')
plt.tight_layout()
plt.show()

print('La caída es pronunciada de K=1 a K=3, y mucho más plana después — el "codo" está cerca')
print('de K=3, que es justo el número real de grupos que usamos para generar los datos.')
print('Como advierte la diapositiva: el codo es ilustrativo y puede ser ambiguo; no es la única señal.')

---
## 4. Coeficiente silhouette

Como en la diapositiva 7: para cada punto, $s = (b-a) / \max(a,b)$, donde $a$ es la distancia media al propio grupo y $b$ la menor distancia media a otro grupo. Cerca de 1 = buena separación; cerca de 0 = en la frontera; negativo = mala asignación.

In [ ]:
puntajes_silhouette = []
for k in range(2, 8):
    modelo = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X_grupos)
    puntaje = silhouette_score(X_grupos, modelo.labels_)
    puntajes_silhouette.append(puntaje)
    print(f'K={k}: silhouette promedio = {puntaje:.3f}')

mejor_k = list(range(2, 8))[np.argmax(puntajes_silhouette)]
print(f'\nMejor K según silhouette: {mejor_k} (coincide con el codo de la sección 3)')

In [ ]:
modelo_final = KMeans(n_clusters=mejor_k, n_init=10, random_state=0).fit(X_grupos)
silhouette_por_punto = silhouette_samples(X_grupos, modelo_final.labels_)

fig, ax = plt.subplots()
y_actual = 0
for grupo in range(mejor_k):
    valores = np.sort(silhouette_por_punto[modelo_final.labels_ == grupo])
    ax.barh(range(y_actual, y_actual + len(valores)), valores, height=1.0)
    y_actual += len(valores)
ax.axvline(silhouette_por_punto.mean(), color='red', linestyle='--', label='Promedio general')
ax.set_xlabel('Coeficiente silhouette')
ax.set_ylabel('Puntos (agrupados por clúster)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Silhouette promedio con K={mejor_k}: {silhouette_por_punto.mean():.3f}')
print(f'¿Algún punto con silhouette negativo (mal asignado)? {(silhouette_por_punto < 0).sum()} de {len(silhouette_por_punto)}')

---
## 5. Inicialización: ¿importa dónde arrancan los centroides?

Como en la diapositiva 5: ninguna alternativa garantiza el mínimo global, pero `k-means++` (que favorece centroides iniciales separados entre sí) suele dar soluciones más consistentes que elegir centroides puramente al azar. Lo comprobamos repitiendo el ajuste 20 veces con cada estrategia.

In [ ]:
inercias_random, inercias_kmeans_pp = [], []
for semilla in range(20):
    modelo_random = KMeans(n_clusters=3, init='random', n_init=1, random_state=semilla).fit(X_grupos)
    modelo_pp = KMeans(n_clusters=3, init='k-means++', n_init=1, random_state=semilla).fit(X_grupos)
    inercias_random.append(modelo_random.inertia_)
    inercias_kmeans_pp.append(modelo_pp.inertia_)

print(f'Init aleatoria:  inercia media={np.mean(inercias_random):8.1f}   desviación estándar={np.std(inercias_random):8.1f}')
print(f'Init k-means++:  inercia media={np.mean(inercias_kmeans_pp):8.1f}   desviación estándar={np.std(inercias_kmeans_pp):8.1f}')

fig, ax = plt.subplots()
ax.boxplot([inercias_random, inercias_kmeans_pp], tick_labels=['random', 'k-means++'])
ax.set_ylabel('Inercia final (20 corridas)')
plt.tight_layout()
plt.show()

print('\nCon inicialización aleatoria, algunas corridas quedan atrapadas en soluciones peores')
print('(inercia más alta y variable). k-means++ es mucho más consistente — por eso es el valor')
print('por defecto en scikit-learn (parámetro init, junto con repetir con varios n_init).')

---
## 6. Alcance y limitaciones de K-means

Como en la diapositiva 8: K-means funciona bien con grupos **compactos y aproximadamente esféricos**, pero falla con formas no convexas y es sensible a valores atípicos.

### 6.1 Grupos no convexos (dos lunas)

In [ ]:
X_lunas, y_lunas_real = make_moons(n_samples=300, noise=0.07, random_state=0)

modelo_lunas_kmeans = KMeans(n_clusters=2, n_init=10, random_state=0).fit(X_lunas)
modelo_lunas_dbscan = DBSCAN(eps=0.2, min_samples=5).fit(X_lunas)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
axes[0].scatter(X_lunas[:, 0], X_lunas[:, 1], c=y_lunas_real, cmap='coolwarm', s=12)
axes[0].set_title('Estructura real (2 lunas)')
axes[1].scatter(X_lunas[:, 0], X_lunas[:, 1], c=modelo_lunas_kmeans.labels_, cmap='coolwarm', s=12)
axes[1].set_title('K-means (K=2)')
axes[2].scatter(X_lunas[:, 0], X_lunas[:, 1], c=modelo_lunas_dbscan.labels_, cmap='coolwarm', s=12)
axes[2].set_title('DBSCAN (alternativa)')
plt.tight_layout()
plt.show()

print('K-means, al buscar grupos esféricos, traza un corte casi vertical que parte las dos lunas')
print('por la mitad — no logra separarlas por su forma real. DBSCAN (una de las alternativas que')
print('menciona la diapositiva 8, junto con spectral clustering) sí reconstruye las dos lunas,')
print('porque no asume que los grupos sean convexos.')

### 6.2 Sensibilidad a valores atípicos

In [ ]:
X_con_atipico = np.vstack([X_grupos, [[50, 50]]])  # un solo punto muy alejado

modelo_sin_atipico = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X_grupos)
modelo_con_atipico = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X_con_atipico)

print('Centroides SIN el valor atípico:')
print(np.round(modelo_sin_atipico.cluster_centers_, 2))
print('\nCentroides CON un solo valor atípico agregado en (50, 50):')
print(np.round(modelo_con_atipico.cluster_centers_, 2))

print('\nComo las medias (el centroide es un promedio) son sensibles a valores extremos,')
print('un solo punto muy alejado puede acaparar su propio grupo y distorsionar los demás')
print('centroides — igual que vimos con la media en los módulos anteriores del curso.')

---
## 7. Para pensar y discutir en clase

1. En la sección 3, el codo aparece cerca de K=3. Si no supiéramos que generamos los datos con 3 grupos reales, ¿qué otra evidencia (de la sección 4) usarías para confirmar que K=3 es una buena elección?
2. En la sección 6.1, ¿qué tipo de datos reales (no sintéticos) crees que podrían tener una forma "curva" como las dos lunas, donde K-means fallaría?
3. En la sección 6.2, ¿cómo tratarías un valor atípico antes de aplicar K-means: lo eliminarías, lo analizarías aparte, o lo dejarías? Justifica según el contexto del problema.

No hay una única respuesta "correcta": lo importante es justificar cada respuesta con evidencia del propio notebook.

---
## Cierre

En este notebook llevamos a código las ideas de las diapositivas [`3.9_Kmeans_agrupamiento_basado_en_centroides.pdf`](3.9_Kmeans_agrupamiento_basado_en_centroides.pdf):

- Una **iteración completa de K-means a mano**, que sirve como hoja de respuestas de la actividad de las diapositivas.
- El **ciclo de Lloyd-Forgy** (inicializar, asignar, actualizar, repetir) implementado desde cero, visualizando la convergencia de los centroides.
- **Inercia** y el **método del codo** para orientar la elección de K.
- El **coeficiente silhouette**, por punto y promedio, como evidencia adicional a la inercia.
- Por qué la **inicialización** importa, y por qué `k-means++` da resultados más consistentes que centroides aleatorios.
- Dos límites reales de K-means: **grupos no convexos** (y DBSCAN como alternativa) y **sensibilidad a valores atípicos**.

### Recursos adicionales
- [Documentación de `sklearn.cluster.KMeans`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)
- [Documentación de `sklearn.metrics.silhouette_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html)
- [Documentación de `sklearn.cluster.DBSCAN`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html)
- [Galería de comparación de algoritmos de clustering de scikit-learn](https://scikit-learn.org/stable/auto_examples/cluster/plot_cluster_comparison.html)